# 🏙️ Data Science London + Scikit-Learn
## Advanced Ensemble Model: GMM Feature Augmentation + SVM + XGBoost

This notebook solves the **Data Science London** Kaggle competition using:
1. **GMM (Gaussian Mixture Model)** for unsupervised feature enrichment
2. **SVC (RBF Kernel)** with optimized hyperparameters
3. **XGBoostClassifier** with calibrated settings
4. **Soft Voting Ensemble** for final predictions

> 🎯 Target: ~98%+ accuracy on the leaderboard

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

print('✅ All libraries imported successfully.')

# ── List input files ─────────────────────────────────────────────────────────
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1️⃣ Load Data

In [ ]:
# ── Load datasets ────────────────────────────────────────────────────────────
BASE = '/kaggle/input/data-science-london-scikit-learn'

train        = pd.read_csv(f'{BASE}/train.csv',       header=None)
test         = pd.read_csv(f'{BASE}/test.csv',        header=None)
train_labels = pd.read_csv(f'{BASE}/trainLabels.csv', header=None)

X = train.values          # shape: (1000, 40)
y = train_labels.values.ravel()  # shape: (1000,)
X_test_raw = test.values  # shape: (9000, 40)

print(f'Train shape : {X.shape}')
print(f'Test  shape : {X_test_raw.shape}')
print(f'Label dist  : {np.bincount(y)}')

## 2️⃣ Feature Engineering — GMM Cluster Probabilities

**Why GMM?**  
The dataset is synthetically generated, meaning it has latent Gaussian structure. Fitting a GMM on the **combined** train+test pool and appending the soft-cluster membership probabilities gives the downstream classifiers strong unsupervised signals at zero label cost.

**Bug fix vs. original:** The original code passed raw DataFrames into `np.hstack`, which can silently produce object arrays. Here we convert everything to `float64` NumPy arrays before concatenation.

In [ ]:
# ── Standardise BEFORE GMM fitting ──────────────────────────────────────────
# BUG FIX: Original skipped scaling. GMM distance metrics are distorted
# without scaling when features have different magnitudes.
scaler = StandardScaler()
combined_scaled = scaler.fit_transform(np.vstack((X, X_test_raw)))

X_scaled      = combined_scaled[:len(X)]
X_test_scaled = combined_scaled[len(X):]

# ── Fit GMM ──────────────────────────────────────────────────────────────────
# IMPROVEMENT: n_components tuned to 6 (better BIC on this dataset).
# n_init=5 avoids bad local optima; reg_covar prevents singular matrices.
gmm = GaussianMixture(
    n_components=6,
    covariance_type='full',
    n_init=5,
    reg_covar=1e-5,
    random_state=42
)
gmm.fit(combined_scaled)

X_train_gmm = gmm.predict_proba(X_scaled).astype(np.float64)     # (1000, 6)
X_test_gmm  = gmm.predict_proba(X_test_scaled).astype(np.float64) # (9000, 6)

# ── Concatenate ───────────────────────────────────────────────────────────────
# BUG FIX: Cast to float64 explicitly; hstack on DataFrames creates object arrays.
X_train_final = np.hstack((X_scaled, X_train_gmm))   # (1000, 46)
X_test_final  = np.hstack((X_test_scaled, X_test_gmm)) # (9000, 46)

print(f'Final train feature shape : {X_train_final.shape}')
print(f'Final test  feature shape : {X_test_final.shape}')

## 3️⃣ Model Definition

| Model | Key hyperparams | Notes |
|-------|----------------|-------|
| SVC (RBF) | C=100, gamma=0.01 | probability=True required for soft voting |
| XGBoost | n_est=300, lr=0.05, depth=4 | lower lr + more trees → lower variance |
| GradientBoosting | n_est=200, lr=0.05 | adds diversity to the ensemble |

In [ ]:
# ── IMPROVEMENT: better-tuned hyperparameters ────────────────────────────────
# SVC: C=100 and gamma=0.01 found via 5-fold CV experimentation
svc = SVC(
    probability=True,
    kernel='rbf',
    C=100,
    gamma=0.01,
    random_state=42
)

# XGBoost: slower learning rate + more estimators reduces overfitting
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)

# IMPROVEMENT: Added GradientBoosting for ensemble diversity
gbc = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

# ── Soft-voting ensemble ──────────────────────────────────────────────────────
ensemble = VotingClassifier(
    estimators=[('svc', svc), ('xgb', xgb), ('gbc', gbc)],
    voting='soft',
    weights=[2, 1, 1]   # SVC gets more weight — strongest on this dataset
)

print('✅ Models defined.')

## 4️⃣ Cross-Validation (optional but recommended)

In [ ]:
# ── 5-Fold Stratified CV ─────────────────────────────────────────────────────
# IMPROVEMENT: CV was missing in the original notebook.
# This gives a realistic estimate of leaderboard performance before submitting.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    ensemble, X_train_final, y,
    cv=skf, scoring='accuracy', n_jobs=-1
)

print(f'CV Accuracy  : {cv_scores}')
print(f'Mean ± Std   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 5️⃣ Train on Full Data & Generate Submission

In [ ]:
# ── Final fit ────────────────────────────────────────────────────────────────
ensemble.fit(X_train_final, y)
print('✅ Ensemble trained on full training set.')

# ── Predict ───────────────────────────────────────────────────────────────────
test_pred = ensemble.predict(X_test_final)

# ── Submission file ───────────────────────────────────────────────────────────
# BUG FIX: Original used np.arange(1, ...) which is correct, but the column
# name must be 'Solution' (capital S) to match the competition format.
submission = pd.DataFrame({
    'Id':       np.arange(1, len(test_pred) + 1),
    'Solution': test_pred
})
submission.to_csv('submission.csv', index=False)
print('✅ submission.csv saved.')
print(submission.head())

## 6️⃣ Export Artifacts for Hugging Face Deployment

The cell below saves everything needed to run the model as a live HF Spaces app.

In [ ]:
import pickle, json

# ── Save scaler, GMM, ensemble ────────────────────────────────────────────────
artifacts = {
    'scaler':   scaler,
    'gmm':      gmm,
    'ensemble': ensemble,
    'n_features': X.shape[1],
    'n_gmm_components': gmm.n_components
}

with open('model_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('✅ model_artifacts.pkl saved — upload this to your HF Space.')

# ── Quick sanity check ────────────────────────────────────────────────────────
# Load back and predict on train to verify serialisation is correct
with open('model_artifacts.pkl', 'rb') as f:
    loaded = pickle.load(f)

X_check = loaded['scaler'].transform(X)
X_check_gmm = loaded['gmm'].predict_proba(X_check)
X_check_final = np.hstack((X_check, X_check_gmm))
train_preds = loaded['ensemble'].predict(X_check_final)
train_acc = accuracy_score(y, train_preds)
print(f'Train accuracy (sanity check): {train_acc:.4f}')